# LumenY 8 — Direction Model (MFE-Filtered Signals)

Binary classifier trained **only on MFE Q50 > 70 bars**.
Predicts direction: long (1) or short (0), based on which way price moved furthest over 72H.

**Target**: `mfe_dir = 1` if `mfe_long_pips > mfe_short_pips`, else `0`  
**Key feature added**: `mfe_q50_oof` (OOF prediction from MFE model) included as feature

Architecture mirrors `notebooks_6/04_meta_confidence_model.ipynb` but uses:
- `features_9` (320 features)
- MFE-filtered population only
- Binary classification (not quantile regression)

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import joblib
import warnings
import gc
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score, confusion_matrix

FEATURES_DIR  = Path('../backend/data/features_9')
MFE_MODEL_DIR = Path('../backend/models_9/mfe_q50')
MODELS_DIR    = Path('../backend/models_9/direction')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# -- Training cutoff (same as MFE model) --
TRAIN_END = '2024-06-30'
MFE_THRESHOLD = 70.0  # filter: only train on bars where OOF mfe_q50 > 70

print('Ready.')
print(f'Direction model: binary classifier on MFE Q50 > {MFE_THRESHOLD} population')
print(f'Training cutoff: {TRAIN_END}')

## 1. Load Dataset

Load features_9. Build `mfe_dir` target (which direction traveled furthest over 72H).

In [ ]:
# Load features_9 (all 15 pairs)
dfs = []
for f in sorted(FEATURES_DIR.glob('*_features.parquet')):
    tmp = pd.read_parquet(f)
    dfs.append(tmp)

df_all = pd.concat(dfs).sort_index()
del dfs
print(f'features_9: {df_all.shape}')
print(f'Pairs: {sorted(df_all["pair"].unique())}')
print(f'Date range: {df_all.index.min().date()} to {df_all.index.max().date()}')

In [ ]:
# Build feature cols (same exclusions as MFE model)
label_cols   = [c for c in df_all.columns if c.startswith('label_')] + [
    'mfe_long_pips', 'mfe_short_pips', 'mfe_atr_24', 'trail_long_bars',
    'trail_short_bars', 'trail_stop_pips'
]
drop_cols    = label_cols + ['pair']
feature_cols = [c for c in df_all.columns if c not in drop_cols]

print(f'Feature columns: {len(feature_cols)}')

# Create mfe_dir target: 1=long won, 0=short won
df_all['mfe_max'] = df_all[['mfe_long_pips', 'mfe_short_pips']].max(axis=1)
df_all['mfe_dir'] = (df_all['mfe_long_pips'] > df_all['mfe_short_pips']).astype(int)

print(f'mfe_dir distribution:')
print(df_all['mfe_dir'].value_counts())
print(f'Balance (1=long): {df_all["mfe_dir"].mean():.3f}')

# Split train/test
df_train = df_all[df_all.index <= TRAIN_END].copy()
df_test  = df_all[df_all.index > TRAIN_END].copy()
print(f'Train: {len(df_train):,} | Test: {len(df_test):,}')

## 2. MFE Model OOF Predictions (Walk-Forward)

Re-run walk-forward CV to generate unbiased `mfe_q50_oof` for the training set.
These OOF preds are then used as a feature for the direction model — no leakage.

In [ ]:
# Load MFE model bundle
mfe_bundle       = joblib.load(MFE_MODEL_DIR / 'model_1H_Q50.joblib')
mfe_feature_cols = mfe_bundle['feature_cols']
mfe_n_iters      = mfe_bundle['n_iters']

print(f'MFE model: {len(mfe_feature_cols)} features, {mfe_n_iters} iterations')

In [ ]:
def walk_forward_splits(n, n_splits=5, test_ratio=0.1):
    test_size = int(n * test_ratio)
    splits = []
    for i in range(n_splits):
        test_start = int(n * 0.5) + i * (int(n * 0.5) // n_splits)
        test_end   = test_start + test_size
        if test_end > n:
            break
        splits.append((list(range(0, test_start)), list(range(test_start, test_end))))
    return splits

y_mfe_all  = df_train['mfe_max']
valid_mask = y_mfe_all.notna()
X_mfe = df_train[mfe_feature_cols][valid_mask].ffill().fillna(0)
y_mfe = y_mfe_all[valid_mask]

splits     = walk_forward_splits(len(X_mfe))
oof_mfe_q50 = np.full(len(X_mfe), np.nan)

mfe_params = {
    'objective': 'quantile', 'alpha': 0.50, 'metric': 'quantile',
    'boosting_type': 'gbdt', 'n_estimators': mfe_n_iters,
    'learning_rate': 0.02, 'num_leaves': 64, 'max_depth': 6,
    'min_child_samples': 50, 'feature_fraction': 0.7,
    'bagging_fraction': 0.8, 'bagging_freq': 5,
    'reg_alpha': 0.1, 'reg_lambda': 0.1,
    'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'device': 'gpu',
}

print(f'Generating MFE OOF predictions ({len(splits)} folds)...')
for fold, (train_idx, test_idx) in enumerate(splits):
    X_tr, y_tr = X_mfe.iloc[train_idx], y_mfe.iloc[train_idx]
    X_te = X_mfe.iloc[test_idx]
    model = lgb.LGBMRegressor(**mfe_params)
    model.fit(X_tr, y_tr, callbacks=[lgb.log_evaluation(-1)])
    oof_mfe_q50[test_idx] = model.predict(X_te)
    print(f'  Fold {fold+1}/{len(splits)}: {len(test_idx):,} samples')
    del model; gc.collect()

# Attach OOF predictions to training rows
df_train_valid = df_train[valid_mask].copy()
df_train_valid['mfe_q50_oof'] = oof_mfe_q50

# Test set: use final MFE model directly
X_test_mfe = df_test[mfe_feature_cols].ffill().fillna(0)
df_test['mfe_q50_oof'] = mfe_bundle['model'].predict(X_test_mfe)

valid_oof_count = (~np.isnan(oof_mfe_q50)).sum()
print(f'OOF coverage: {valid_oof_count:,} / {len(oof_mfe_q50):,}')
print(f'OOF mfe_q50 mean: {np.nanmean(oof_mfe_q50):.2f}')

## 3. Filter to MFE Q50 > 70 Population

Only train direction model on bars where MFE signal is strong.
This mirrors the simulation gate exactly.

In [ ]:
# Filter: only MFE Q50 > 70 bars
df_dir_train = df_train_valid[df_train_valid['mfe_q50_oof'] > MFE_THRESHOLD].copy()
df_dir_test  = df_test[df_test['mfe_q50_oof'] > MFE_THRESHOLD].copy()

print(f'=== MFE-Filtered Population ===')
print(f'Train: {len(df_dir_train):,} bars ({len(df_dir_train)/len(df_train_valid)*100:.1f}% of train)')
print(f'Test:  {len(df_dir_test):,} bars ({len(df_dir_test)/len(df_test)*100:.1f}% of test)')
print(f'Train mfe_dir balance: {df_dir_train["mfe_dir"].mean():.3f} (1=long)')
print(f'Test  mfe_dir balance: {df_dir_test["mfe_dir"].mean():.3f} (1=long)')

## 4. Walk-Forward CV — Direction Classifier

In [ ]:
# Feature set: features_9 columns + mfe_q50_oof
dir_feature_cols = feature_cols + ['mfe_q50_oof']
TARGET_DIR = 'mfe_dir'

dir_params = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'boosting_type': 'gbdt', 'n_estimators': 3000,
    'learning_rate': 0.02, 'num_leaves': 64, 'max_depth': 6,
    'min_child_samples': 30, 'feature_fraction': 0.7,
    'bagging_fraction': 0.8, 'bagging_freq': 5,
    'reg_alpha': 0.1, 'reg_lambda': 0.1,
    'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'device': 'gpu',
}

y_dir_all = df_dir_train[TARGET_DIR]
valid_dir = y_dir_all.notna()
X_dir = df_dir_train[dir_feature_cols][valid_dir].ffill().fillna(0)
y_dir = y_dir_all[valid_dir]

splits_dir    = walk_forward_splits(len(X_dir))
oof_dir_prob  = np.full(len(X_dir), np.nan)
best_iters_dir = []

print(f'Training direction classifier on {len(X_dir):,} samples ({len(splits_dir)} folds)...')
for fold, (train_idx, test_idx) in enumerate(splits_dir):
    X_tr, y_tr = X_dir.iloc[train_idx], y_dir.iloc[train_idx]
    X_te, y_te = X_dir.iloc[test_idx], y_dir.iloc[test_idx]

    model = lgb.LGBMClassifier(**dir_params)
    model.fit(X_tr, y_tr,
              eval_set=[(X_te, y_te)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])

    oof_dir_prob[test_idx] = model.predict_proba(X_te)[:, 1]
    best_iters_dir.append(model.best_iteration_)
    acc_f = accuracy_score(y_te, (oof_dir_prob[test_idx] > 0.5).astype(int))
    try:
        auc_f = roc_auc_score(y_te, oof_dir_prob[test_idx])
    except Exception:
        auc_f = float('nan')
    print(f'  Fold {fold+1}: acc={acc_f:.4f}, AUC={auc_f:.4f}, best_iter={model.best_iteration_}')
    del model; gc.collect()

print(f'Avg best iter: {np.mean(best_iters_dir):.0f}')

## 5. OOF Evaluation

In [ ]:
valid_oof_dir = ~np.isnan(oof_dir_prob)
y_oof_dir = y_dir.values[valid_oof_dir]
p_oof     = oof_dir_prob[valid_oof_dir]

acc_oof = accuracy_score(y_oof_dir, (p_oof > 0.5).astype(int))
auc_oof = roc_auc_score(y_oof_dir, p_oof)

print(f'=== OOF Evaluation ===')
print(f'Accuracy (>0.50 threshold): {acc_oof:.4f} ({acc_oof*100:.1f}%)')
print(f'ROC-AUC:                    {auc_oof:.4f}')
print()

# Threshold sweep
print(f'Threshold sweep (OOF):')
print(f'{"Threshold":>14} | {"N Signals":>10} | {"Coverage":>9} | {"Accuracy":>9}')
print('-' * 52)
for thr in [0.45, 0.50, 0.52, 0.55, 0.58, 0.60, 0.65, 0.70]:
    mask_long  = p_oof >= thr
    mask_short = p_oof <= (1 - thr)
    mask = mask_long | mask_short
    if mask.sum() < 100:
        continue
    pred_t = np.where(mask_long[mask], 1, 0)
    acc_t  = accuracy_score(y_oof_dir[mask], pred_t)
    print(f'p>{thr:.2f} or <{1-thr:.2f}   | {mask.sum():>10,} ({mask.mean()*100:.1f}%) | {acc_t:.4f}')

## 6. Final Model Training + Test Evaluation

In [ ]:
avg_iter = max(50, int(np.mean(best_iters_dir)))
params_final = {**dir_params, 'n_estimators': avg_iter}

print(f'Training final direction model ({avg_iter} iterations)...')
final_dir_model = lgb.LGBMClassifier(**params_final)
final_dir_model.fit(X_dir, y_dir, callbacks=[lgb.log_evaluation(-1)])
print('Done.')

# Save
save_path = MODELS_DIR / 'direction_model.joblib'
joblib.dump({
    'model':            final_dir_model,
    'feature_cols':     dir_feature_cols,
    'mfe_feature_cols': mfe_feature_cols,
    'train_end':        TRAIN_END,
    'mfe_threshold':    MFE_THRESHOLD,
    'n_iters':          avg_iter,
    'task':             'direction_binary',
}, save_path)
size_mb = save_path.stat().st_size / 1024 / 1024
print(f'Saved ({size_mb:.1f} MB) -> {save_path}')

In [ ]:
X_test_dir   = df_dir_test[dir_feature_cols].ffill().fillna(0)
y_test_dir   = df_dir_test[TARGET_DIR]
valid_test   = y_test_dir.notna()
X_test_clean = X_test_dir[valid_test]
y_test_clean = y_test_dir[valid_test]

test_probs = final_dir_model.predict_proba(X_test_clean)[:, 1]
test_preds = (test_probs > 0.5).astype(int)

acc_test = accuracy_score(y_test_clean, test_preds)
auc_test = roc_auc_score(y_test_clean, test_probs)

print(f'=== Test Set Evaluation (>{TRAIN_END}) ===')
print(f'Samples:  {len(y_test_clean):,}')
print(f'Accuracy: {acc_test:.4f} ({acc_test*100:.1f}%)')
print(f'ROC-AUC:  {auc_test:.4f}')

results = pd.DataFrame({
    'actual_dir': y_test_clean.values,
    'prob_long':  test_probs,
    'pred_dir':   test_preds,
    'pair':       df_dir_test.loc[valid_test, 'pair'].values,
    'mfe_q50':    df_dir_test.loc[valid_test, 'mfe_q50_oof'].values,
    'mfe_long':   df_dir_test.loc[valid_test, 'mfe_long_pips'].values,
    'mfe_short':  df_dir_test.loc[valid_test, 'mfe_short_pips'].values,
}, index=y_test_clean.index)

cm = confusion_matrix(y_test_clean, test_preds)
print(f'\nConfusion matrix (rows=actual, cols=pred):')
print(f'           Pred Long | Pred Short')
print(f'Act Long   {cm[1,1]:>9,} | {cm[1,0]:>9,}')
print(f'Act Short  {cm[0,1]:>9,} | {cm[0,0]:>9,}')

## 7. Threshold Sweep (Test Set)

In [ ]:
print(f'=== Threshold Sweep (Test Set) ===')
print(f'{"Threshold":>14} | {"N Signals":>10} | {"Coverage":>9} | {"Accuracy":>9}')
print('-' * 52)
for thr in [0.45, 0.50, 0.52, 0.55, 0.58, 0.60, 0.65, 0.70]:
    mask_long  = test_probs >= thr
    mask_short = test_probs <= (1 - thr)
    mask = mask_long | mask_short
    if mask.sum() < 30:
        continue
    pred_t   = np.where(mask_long[mask], 1, 0)
    acc_t    = accuracy_score(y_test_clean.values[mask], pred_t)
    print(f'p>{thr:.2f} or <{1-thr:.2f}   | {mask.sum():>10,} ({mask.mean()*100:.1f}%) | {acc_t:.4f}')

## 8. Per-Pair Breakdown

In [ ]:
print(f'=== Per-Pair Direction Accuracy (Test Set) ===')
print(f'{"Pair":>8} | {"N":>7} | {"Acc":>7} | {"AUC":>7} | {"Long%":>7}')
print('-' * 50)
for pair in sorted(results['pair'].unique()):
    sub = results[results['pair'] == pair]
    if len(sub) < 20:
        continue
    acc_p = accuracy_score(sub['actual_dir'], sub['pred_dir'])
    try:
        auc_p = roc_auc_score(sub['actual_dir'], sub['prob_long'])
    except Exception:
        auc_p = float('nan')
    long_rate = sub['actual_dir'].mean()
    print(f'{pair:>8} | {len(sub):>7,} | {acc_p:.4f} | {auc_p:.4f} | {long_rate:.3f}')

## 9. Feature Importance

In [ ]:
importances = pd.Series(
    final_dir_model.feature_importances_,
    index=dir_feature_cols
).sort_values(ascending=False)

print('Top 30 features:')
print(importances.head(30).to_string())

fig, ax = plt.subplots(figsize=(10, 10))
fig.patch.set_facecolor('#080c14')
ax.set_facecolor('#080c14')
importances.head(30).plot(kind='barh', ax=ax, color='#00d4ff')
ax.invert_yaxis()
ax.set_title('Top 30 Features — Direction Model', color='white')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#334455')
plt.tight_layout()
plt.show()

## 10. Summary

In [ ]:
print('=' * 70)
print('DIRECTION MODEL — TRAINING COMPLETE')
print('=' * 70)
print(f'Population:    MFE Q50 > {MFE_THRESHOLD} pips (strongly moving bars only)')
print(f'Train samples: {len(X_dir):,}')
print(f'Test samples:  {len(y_test_clean):,}')
print(f'Target:        mfe_dir (1=long, 0=short based on 72H MFE)')
print(f'Features:      {len(dir_feature_cols)} (features_9 + mfe_q50_oof)')
print()
print(f'OOF  Accuracy: {acc_oof:.4f} ({acc_oof*100:.1f}%)')
print(f'Test Accuracy: {acc_test:.4f} ({acc_test*100:.1f}%)')
print(f'Test ROC-AUC:  {auc_test:.4f}')
print()
print(f'Model saved:   {save_path}')
print(f'Iterations:    {avg_iter}')
print('=' * 70)